# 02 — Pilot SAE Training on DINOv2 Activations

Trains a BatchTopK SAE on the cached DINOv2 ViT-B/14 layer-11 activations as a sanity check.

**Pilot config:** 1.28M images × 256 patches = 327.68M patch tokens, 16× expansion (dict_size=12288), k=192.

In [4]:
# ── Cell 1: Load stats and discover shards ────────────────────────────────────
import glob
import json
import os
import sys

import torch

# Make sure the repo root is on the path when running from notebooks/
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src.utils.paths import ACTIVATION_ROOT

ACTIVATION_DIR = os.path.join(ACTIVATION_ROOT, "dinov2_vitb14", "layer_11")

# Load stats
stats_path = os.path.join(ACTIVATION_DIR, "stats.json")
with open(stats_path) as f:
    stats = json.load(f)

mean = torch.tensor(stats["mean"], dtype=torch.float32)  # [768]
std  = float(stats["std"])                                # scalar
print(f"Stats loaded — mean shape: {mean.shape},  std: {std:.4f}")
print(f"Backbone: {stats.get('backbone', 'dinov2_vitb14')}  "
      f"layer: {stats.get('layer', 11)}  "
      f"num_images: {stats.get('num_images')}  "
      f"d_model: {stats.get('d_model')}")

# Discover shards — do NOT load them yet
shard_paths = sorted(glob.glob(os.path.join(ACTIVATION_DIR, "shard_*.pt")))
print(f"\nFound {len(shard_paths)} shard(s)")
print("Shards will be loaded one at a time during training to stay within RAM limits.")

Stats loaded — mean shape: torch.Size([768]),  std: 1.8916
Backbone: dinov2_vitb14  layer: 11  num_images: 1281167  d_model: 768

Found 257 shard(s)
Shards will be loaded one at a time during training to stay within RAM limits.


In [2]:
# ── Cell 2: Configure and instantiate BatchTopK SAE ───────────────────────────
from overcomplete import BatchTopKSAE, TopKSAE
from src.training.overcomplete_config import make_sae_config

D_MODEL          = 768
EXPANSION_FACTOR = 16
DICT_SIZE        = D_MODEL * EXPANSION_FACTOR  # 12288
K                = 192   # active features per batch step
THRESHOLD_MOM    = 0.9
BATCH_SIZE = 4096

device = "cuda" if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else "cpu"
print(f"Device: {device}")

config = make_sae_config(
    d_model=D_MODEL,
    expansion_factor=EXPANSION_FACTOR,
    k=K,
    architecture="batchtopk",
    batch_size=BATCH_SIZE,
)
print(f"Config: {config}")

# Instantiate via overcomplete and call .tied() to use dictionary^T as the encoder.
# The default MLPEncoder uses BatchNorm1d(12288) which is very slow on CPU/MPS.
# Tied mode is standard SAE practice and avoids this overhead entirely.
sae = BatchTopKSAE(
    input_shape=D_MODEL,
    nb_concepts=DICT_SIZE,
    top_k=K * BATCH_SIZE,
    device=device
)

num_params = sum(p.numel() for p in sae.parameters())
print(f"\nBatchTopKSAE instantiated:")
print(f"  input_shape:  {D_MODEL}")
print(f"  nb_concepts:  {DICT_SIZE}  ({EXPANSION_FACTOR}× expansion)")
print(f"  top_k:        {K}")
print(f"  parameters:   {num_params:,}")

Device: cuda
Config: {'architecture': 'batchtopk', 'd_model': 768, 'expansion_factor': 16, 'dict_size': 12288, 'k': 192, 'constructor_kwargs': {'input_shape': 768, 'nb_concepts': 12288, 'top_k': 786432}}

BatchTopKSAE instantiated:
  input_shape:  768
  nb_concepts:  12288  (16× expansion)
  top_k:        192
  parameters:   18,886,656


In [6]:
# ── Cell 3: Train the SAE (shard-by-shard to stay within RAM) ─────────────────
import random
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

BATCH_SIZE  = BATCH_SIZE
LR          = 3e-4
NUM_EPOCHS  = 2
LOG_EVERY   = 50

optimizer   = torch.optim.Adam(sae.parameters(), lr=LR)
training_log = []
global_step  = 0

# Estimate steps for progress reporting
approx_tokens_per_shard = 5000 * 256  # shard_size * patches
steps_per_shard = (approx_tokens_per_shard + BATCH_SIZE - 1) // BATCH_SIZE
total_steps = NUM_EPOCHS * len(shard_paths) * steps_per_shard
print(f"Training: {NUM_EPOCHS} epochs × {len(shard_paths)} shards  (~{total_steps} steps estimated)")
print(f"Batch size: {BATCH_SIZE}  |  LR: {LR}")
print("Each shard is loaded, trained on, then freed — max RAM use ~3 GB per shard.\n")

sae.train()
for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    epoch_steps = 0

    # Shuffle shard order each epoch for better coverage
    epoch_shards = shard_paths[:]
    random.shuffle(epoch_shards)

    for shard_path in tqdm(epoch_shards, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} shards", leave=False):
        # Load shard, normalize, flatten, shuffle
        shard = torch.load(shard_path, map_location="cpu", weights_only=True)  # [N, 256, 768]
        N, P, D = shard.shape
        tokens = shard.reshape(N * P, D).float()
        tokens = (tokens - mean) / std
        perm = torch.randperm(tokens.shape[0])
        tokens = tokens[perm]

        loader = DataLoader(TensorDataset(tokens), batch_size=BATCH_SIZE,
                            shuffle=False, drop_last=False, num_workers=0)
        for (batch,) in loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            _pre_codes, codes, x_hat = sae(batch)
            loss = (batch - x_hat).square().mean()
            loss.backward()
            optimizer.step()

            loss_val     = float(loss.item())
            epoch_loss  += loss_val
            epoch_steps += 1
            global_step += 1

            if global_step % LOG_EVERY == 0:
                training_log.append({"step": global_step, "loss": loss_val})

        del shard, tokens, loader

    avg = epoch_loss / epoch_steps if epoch_steps > 0 else float("nan")
    print(f"Epoch {epoch+1:>2}/{NUM_EPOCHS}  |  step {global_step:>6}  |  avg loss: {avg:.4f}")

print("\nTraining complete.")

Training: 2 epochs × 257 shards  (~160882 steps estimated)
Batch size: 4096  |  LR: 0.0003
Each shard is loaded, trained on, then freed — max RAM use ~3 GB per shard.



Epoch  1/2  |  step  80201  |  avg loss: 0.0842


Epoch  2/2  |  step 160402  |  avg loss: 0.0802

Training complete.


In [7]:
# ── Cell 4: Save checkpoint (before eval — so weights are safe if eval OOMs) ───
import yaml

from src.utils.paths import CHECKPOINT_ROOT

# CHECKPOINT_DIR = os.path.join(CHECKPOINT_ROOT, "dinov2_vitb14", "batchtopk_16x_k192")
CHECKPOINT_DIR = os.path.join("pilot_full_run", "dinov2_vitb14", "batchtopk_16x_k192")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

sae_path = os.path.join(CHECKPOINT_DIR, "sae.pt")
# Include running_threshold for BatchTopKSAE — it's a plain attribute,
# not a registered buffer, so state_dict() doesn't capture it.
save_dict = sae.state_dict()
if hasattr(sae, "running_threshold") and sae.running_threshold is not None:
    save_dict["_running_threshold"] = sae.running_threshold.detach().cpu()
torch.save(save_dict, sae_path)
print(f"Saved SAE weights  → {sae_path}")

cfg = {
    "backbone":         "dinov2_vitb14",
    "architecture":     "batchtopk",
    "d_model":          D_MODEL,
    "expansion_factor": EXPANSION_FACTOR,
    "dict_size":        DICT_SIZE,
    "k":                K,
    "lr":               LR,
    "batch_size":       BATCH_SIZE,
    "num_epochs":       NUM_EPOCHS,
    "total_steps":      global_step,
    "activation_dir":   ACTIVATION_DIR,
}
config_path = os.path.join(CHECKPOINT_DIR, "config.yaml")
with open(config_path, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)
print(f"Saved config       → {config_path}")

log_path = os.path.join(CHECKPOINT_DIR, "training_log_1e_3_ep2.json")
with open(log_path, "w") as f:
    json.dump({"training_loss": training_log}, f, indent=2)
print(f"Saved training log → {log_path}")
print("\nCheckpoint saved. Run Cell 5 to evaluate.")

Saved SAE weights  → pilot_full_run/dinov2_vitb14/batchtopk_16x_k192/sae.pt
Saved config       → pilot_full_run/dinov2_vitb14/batchtopk_16x_k192/config.yaml
Saved training log → pilot_full_run/dinov2_vitb14/batchtopk_16x_k192/training_log_1e_3_ep2.json

Checkpoint saved. Run Cell 5 to evaluate.


In [8]:
# ── Cell 5: Run ALL metrics (M1–M7) on validation shards ──────────────────────
import time
import traceback

import numpy as np
from tqdm.auto import tqdm

from scripts.run_all_metrics import (
    encode_val_data,
    run_m1_fvu,
    run_m2_downstream,
    run_m3_sparse_probing,
    run_m4_monosemanticity,
    run_m5_cross_domain,
    run_m6_localization,
    run_m7_absorption,
    format_time,
)
from src.evaluation.concept_detection.sparse_probing import (
    discover_shards,
    load_imagenet_val_labels,
    load_stats,
)

# ── Configuration ─────────────────────────────────────────────────────────────
BACKBONE_NAME    = "dinov2_vitb14"
ENCODE_BATCH     = 512
MAX_FEATURES_M4  = 2048
M5_DATASETS      = ["eurosat"]

# Use val activation shards
val_shard_dir = '/mnt/NAS/data/ds5725/visaebench/activations_val/dinov2_vitb14/layer_11'
eval_shard_paths = discover_shards(val_shard_dir)
val_mean, val_std = load_stats(val_shard_dir)
print(f"Val shards: {len(eval_shard_paths)} from {val_shard_dir}")

# ── Load ImageNet val labels (needed by M2, M3, M7) ──────────────────────────
print("Loading ImageNet val labels...")
labels = load_imagenet_val_labels()
print(f"{len(labels)} labels loaded")

# ── Shared encoding pass (M1 FVU piggy-backed) ───────────────────────────────
sae.eval()
print("\n--- Shared Encoding Pass (+ M1 FVU) ---")
t0 = time.time()
image_codes, pooled_original, pooled_reconstructed, fvu_stats = encode_val_data(
    sae=sae,
    shard_paths=eval_shard_paths,
    mean=val_mean,
    std=val_std,
    device=device,
    dict_size=DICT_SIZE,
    encode_batch_size=ENCODE_BATCH,
    compute_fvu=True,
)
enc_time = time.time() - t0
print(f"Shared encoding done: {image_codes.shape[0]} images in {format_time(enc_time)}")
print(f"  image_codes:          {image_codes.shape}")
print(f"  pooled_original:      {pooled_original.shape}")
print(f"  pooled_reconstructed: {pooled_reconstructed.shape}")

# ── Run each metric ──────────────────────────────────────────────────────────
metric_order = ["m1", "m2", "m3", "m4", "m5", "m6", "m7"]
metric_labels = {
    "m1": "M1 FVU", "m2": "M2 Downstream", "m3": "M3 Sparse Probing",
    "m4": "M4 Monosemanticity", "m5": "M5 Cross-Domain",
    "m6": "M6 Localization", "m7": "M7 Absorption",
}
summary = {}
all_results = {}

for m in tqdm(metric_order, desc="Running metrics", unit="metric"):
    label = metric_labels[m]
    print(f"\n--- {label} ---")
    t0 = time.time()
    try:
        if m == "m1":
            result = fvu_stats  # already computed during shared encoding
            print("  (from shared encoding pass)")
        elif m == "m2":
            result = run_m2_downstream(
                sae, eval_shard_paths, val_mean, val_std, device, DICT_SIZE,
                labels.copy(), pooled_original, pooled_reconstructed, ENCODE_BATCH,
            )
        elif m == "m3":
            result = run_m3_sparse_probing(image_codes, labels.copy(), DICT_SIZE)
        elif m == "m4":
            result = run_m4_monosemanticity(
                sae, eval_shard_paths, val_mean, val_std, device, DICT_SIZE,
                BACKBONE_NAME, image_codes, MAX_FEATURES_M4,
            )
        elif m == "m5":
            result = run_m5_cross_domain(
                sae, eval_shard_paths, val_mean, val_std, device, DICT_SIZE,
                BACKBONE_NAME, M5_DATASETS,
            )
        elif m == "m6":
            result = run_m6_localization(
                sae, eval_shard_paths, val_mean, val_std, device, DICT_SIZE,
                ENCODE_BATCH,
            )
        elif m == "m7":
            result = run_m7_absorption(image_codes, labels.copy(), DICT_SIZE)

        elapsed = time.time() - t0
        all_results[m] = result
        summary[label] = (result, elapsed)
        print(f"  Done in {format_time(elapsed)}")
    except Exception as e:
        elapsed = time.time() - t0
        print(f"  ERROR: {e}")
        traceback.print_exc()
        summary[label] = ({"error": str(e)}, elapsed)

# ── Summary ──────────────────────────────────────────────────────────────────
total_time = sum(t for _, t in summary.values())
print("\n" + "=" * 60)
print("Evaluation Summary")
print("=" * 60)

if "m1" in all_results:
    r = all_results["m1"]
    print(f"  M1 FVU:              {r['fvu']:.4f}  (L0={r['l0']:.1f}, dead={r['dead_pct']:.1f}%)")
if "m2" in all_results:
    r = all_results["m2"]
    print(f"  M2 Downstream:       ratio={r['preservation_ratio']:.3f}  (orig={r['accuracy_original']:.3f}, recon={r['accuracy_reconstructed']:.3f})")
if "m3" in all_results:
    r = all_results["m3"]
    print(f"  M3 Sparse Probing:   AUC={r['auc']:.3f}  (k32={r.get('k_32_accuracy', 0):.3f}, k128={r.get('k_128_accuracy', 0):.3f})")
if "m4" in all_results:
    r = all_results["m4"]
    print(f"  M4 Monosemanticity:  {r.get('monosemanticity_score', 'N/A')}")
if "m5" in all_results:
    r = all_results["m5"]
    for ds in M5_DATASETS:
        if ds in r:
            print(f"  M5 Cross-Domain:     {ds} preservation_k128={r[ds].get('preservation_k128', 'N/A')}")
if "m6" in all_results:
    r = all_results["m6"]
    mi = r.get("results", {}).get("mean_morans_i")
    print(f"  M6 Localization:     Moran's I={mi}")
if "m7" in all_results:
    r = all_results["m7"]
    print(f"  M7 Absorption:       rate={r['absorption_rate']:.3f}  (tests={r['num_tests']}, absorbed={r['num_absorbed']})")

print("-" * 60)
print(f"  Total time: {format_time(total_time)}")
print("=" * 60)

# ── Save all results to checkpoint dir ────────────────────────────────────────
all_metrics_path = os.path.join(CHECKPOINT_DIR, "all_metrics.json")

def _to_serializable(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

serializable = json.loads(json.dumps(all_results, default=_to_serializable))
with open(all_metrics_path, "w") as f:
    json.dump(serializable, f, indent=2)
print(f"\nAll metrics saved → {all_metrics_path}")

Val shards: 10 from /mnt/NAS/data/ds5725/visaebench/activations_val/dinov2_vitb14/layer_11
Loading ImageNet val labels...
50000 labels loaded

--- Shared Encoding Pass (+ M1 FVU) ---


Shared encoding: 100%|██████████| 50000/50000 [06:25<00:00, 129.76img/s]


Shared encoding done: 50000 images in 6min
  image_codes:          (50000, 12288)
  pooled_original:      (50000, 768)
  pooled_reconstructed: (50000, 768)


Running metrics:   0%|          | 0/7 [00:00<?, ?metric/s]


--- M1 FVU ---
  (from shared encoding pass)
  Done in 0s

--- M2 Downstream ---
[M2] Training 2 probes in parallel...


...

Running metrics:  86%|████████▌ | 6/7 [35:15<05:39, 339.13s/metric]

  Done in 7min

--- M7 Absorption ---
[M7] Using precomputed codes: (50000, 12288)
[hierarchy] WordNet unavailable (No module named 'nltk'), using hardcoded groups
[hierarchy] Hardcoded: 26 groups (265 classes)
[M7] 26 sibling groups


/mnt/NAS/home/ds5725/visaebench-internal/.venv/lib/python3.10/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [    4    39    79    80    83    98   101   108   111   129   155   213
   217   233   290   311   324   414   417   484   487   531   549   553
   563   590   603   620   621   640   675   690   696   755   766   778
   801   822   824   836   871   886   887   917   920   941   956   964
   974   980   994  1046  1064  1082  1091  1117  1188  1273  1297  1300
  1308  1355  1387  1460  1475  1493  1531  1540  1541  1548  1552  1593
  1643  1654  1666  1669  1724  1725  1729  1763  1799  1841  1869  1871
  1895  1927  1941  1942  1946  2014  2032  2061  2082  2127  2161  2212
  2225  2258  2286  2290  2360  2378  2386  2404  2416  2442  2473  2499
  2510  2615  2639  2644  2647  2761  2793  2832  2849  2853  2878  2911
  2923  2936  3006  3011  3027  3046  3127  3155  3196  3220  3319  3412
  3494  3502  3552  3708  3711  3770  3771 

  Done in 11s

Evaluation Summary
  M1 FVU:              0.0878  (L0=206.6, dead=1.1%)
  M2 Downstream:       ratio=1.000  (orig=0.760, recon=0.761)
  M3 Sparse Probing:   AUC=0.541  (k32=0.130, k128=0.415)
  M4 Monosemanticity:  0.8579086732468858
  M5 Cross-Domain:     eurosat preservation_k128=0.9797673242286292
  M6 Localization:     Moran's I=0.22443621765203461
  M7 Absorption:       rate=0.343  (tests=265, absorbed=91)
------------------------------------------------------------
  Total time: 35min


ValueError: Circular reference detected

In [ ]:
# # ── Cell 6 (optional): Load saved checkpoint to resume eval without retraining ─
# # Run this cell instead of Cell 3+4 if the kernel crashed after training was done.
# import yaml

# from src.utils.paths import CHECKPOINT_ROOT

# CHECKPOINT_DIR = os.path.join(CHECKPOINT_ROOT, "dinov2_vitb14", "batchtopk_16x_k192")
# sae_path       = os.path.join(CHECKPOINT_DIR, "sae.pt")
# log_path       = os.path.join(CHECKPOINT_DIR, "training_log.json")

# state_dict = torch.load(sae_path, map_location=device, weights_only=True)

# # Restore running_threshold for BatchTopKSAE (saved under special key)
# saved_threshold = state_dict.pop("_running_threshold", None)
# sae.load_state_dict(state_dict)
# if saved_threshold is not None:
#     sae.running_threshold = saved_threshold.to(device)
# else:
#     # Old checkpoint — calibrate with a dummy forward pass
#     sae.train()
#     with torch.no_grad():
#         sae(torch.randn(BATCH_SIZE, D_MODEL, device=device))
# sae.eval()
# print(f"Loaded checkpoint from {sae_path}")

# with open(os.path.join(CHECKPOINT_DIR, "config.yaml")) as f:
#     saved_cfg = yaml.safe_load(f)

# global_step  = saved_cfg.get("total_steps", 0)
# training_log = []
# print(f"Resumed: global_step={global_step},  device={device}")
# print("Now run Cell 5 to evaluate.")